# 29 — Vision Transformers and Hybrid CNN–Transformer Models

Vision Transformers (ViTs) convert images into patch tokens and use self-attention to model relationships between them.

We will study:

- Image patches
- Patch embeddings
- Positional embeddings
- Self-attention
- Multi-head attention
- Transformer encoder blocks
- Vision Transformer architecture
- CNN vs ViT inductive biases
- Pretrained ViTs in `torchvision`
- Grayscale ultrasound adaptation
- Fine-tuning
- Hybrid CNN–Transformer models


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

print("PyTorch:",torch.__version__)


# 1. From Image to Patches

For image size $(H,W)$ and square patch size $P$:

$$
\boxed{N=\frac{H}{P}\frac{W}{P}}
$$


In [ ]:
H=W=224;P=16
print((H//P)*(W//P))


# 2. Patch Embedding

A convolution with kernel=stride=patch size creates patch embeddings efficiently.


In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self,in_channels=1,patch_size=8,embed_dim=64):
        super().__init__()
        self.proj=nn.Conv2d(in_channels,embed_dim,patch_size,stride=patch_size)

    def forward(self,x):
        x=self.proj(x)
        return x.flatten(2).transpose(1,2)


In [ ]:
x=torch.randn(4,1,64,64)
pe=PatchEmbedding()
tokens=pe(x)
print(tokens.shape)


# 3. Positional Embeddings

Attention does not know spatial order by itself.

Add:

$$
token_i+position_i
$$


# 4. Class Token

Many ViTs prepend a learned `[CLS]` token whose final representation is used for classification.


# 5. Self-Attention

$$
\boxed{
Attention(Q,K,V)=softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
}
$$


In [ ]:
def scaled_dot_product_attention(q,k,v):
    scores=q@k.transpose(-2,-1)/math.sqrt(q.shape[-1])
    w=torch.softmax(scores,dim=-1)
    return w@v,w


# 6. Multi-Head Attention

Multiple heads can learn different token relationships.


In [ ]:
mha=nn.MultiheadAttention(64,4,batch_first=True)
out,w=mha(tokens,tokens,tokens)
print(out.shape)


# 7. Transformer Encoder Block


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self,dim=64,heads=4,mlp_ratio=4):
        super().__init__()
        self.norm1=nn.LayerNorm(dim)
        self.attn=nn.MultiheadAttention(dim,heads,batch_first=True)
        self.norm2=nn.LayerNorm(dim)
        self.mlp=nn.Sequential(
            nn.Linear(dim,dim*mlp_ratio),
            nn.GELU(),
            nn.Linear(dim*mlp_ratio,dim)
        )

    def forward(self,x):
        y=self.norm1(x)
        a,_=self.attn(y,y,y)
        x=x+a
        return x+self.mlp(self.norm2(x))


# 8. Minimal Grayscale ViT


In [ ]:
class MiniViT(nn.Module):
    def __init__(self,image_size=64,patch_size=8,dim=64,depth=4,heads=4,num_classes=3):
        super().__init__()
        assert image_size%patch_size==0
        self.patch=PatchEmbedding(1,patch_size,dim)
        n=(image_size//patch_size)**2
        self.cls=nn.Parameter(torch.zeros(1,1,dim))
        self.pos=nn.Parameter(torch.zeros(1,n+1,dim))
        self.blocks=nn.ModuleList([TransformerBlock(dim,heads) for _ in range(depth)])
        self.norm=nn.LayerNorm(dim)
        self.head=nn.Linear(dim,num_classes)

    def forward(self,x):
        x=self.patch(x)
        cls=self.cls.expand(x.size(0),-1,-1)
        x=torch.cat([cls,x],dim=1)
        x=x+self.pos[:,:x.size(1)]
        for b in self.blocks:
            x=b(x)
        x=self.norm(x)
        return self.head(x[:,0])


In [ ]:
vit=MiniViT()
print(vit(torch.randn(8,1,64,64)).shape)


# 9. Shape Reasoning

For 64×64 with 8×8 patches:

$$
8\times8=64
$$

patch tokens.

With `[CLS]`:

$$
(N,65,D)
$$


# 10. CNN vs ViT

CNNs provide strong local/translation inductive biases.

ViTs rely more heavily on learned relationships and often benefit strongly from large-scale pretraining.


# 11. Pretrained ViT in `torchvision`


In [ ]:
weights=models.ViT_B_16_Weights.DEFAULT
print(weights)


In [ ]:
def build_pretrained_vit(num_classes=3):
    weights=models.ViT_B_16_Weights.DEFAULT
    try:
        model=models.vit_b_16(weights=weights)
        loaded=True
    except Exception as e:
        print("Weights unavailable:",e)
        model=models.vit_b_16(weights=None)
        loaded=False
    in_features=model.heads.head.in_features
    model.heads.head=nn.Linear(in_features,num_classes)
    return model,loaded


# 12. Grayscale Strategy A — Repeat to RGB


In [ ]:
gray=torch.randn(4,1,224,224)
rgb_like=gray.repeat(1,3,1,1)
print(rgb_like.shape)


# 13. Grayscale Strategy B — Modify Patch Projection

Average pretrained RGB weights over channels.


In [ ]:
def convert_vit_to_grayscale(model):
    conv=model.conv_proj
    new=nn.Conv2d(
        1,conv.out_channels,
        kernel_size=conv.kernel_size,
        stride=conv.stride,
        padding=conv.padding,
        bias=(conv.bias is not None)
    )
    with torch.no_grad():
        new.weight.copy_(conv.weight.mean(dim=1,keepdim=True))
        if conv.bias is not None:
            new.bias.copy_(conv.bias)
    model.conv_proj=new
    return model


# 14. Fine-Tuning Strategy

1. Train new classifier head.
2. Unfreeze final transformer blocks.
3. Use lower LR for pretrained layers.


In [ ]:
def freeze_vit_except_head(model):
    for p in model.parameters():
        p.requires_grad=False
    for p in model.heads.parameters():
        p.requires_grad=True


# 15. Parameter Groups


In [ ]:
def fine_tune_groups(model):
    return [
        {"params":model.encoder.layers[-2:].parameters(),"lr":1e-5},
        {"params":model.heads.parameters(),"lr":1e-4},
    ]


# 16. Patch Size Tradeoff

Smaller patches:

- More tokens
- More detail
- Higher compute

Larger patches:

- Fewer tokens
- Lower compute
- Coarser representation


# 17. Attention Complexity

Standard attention grows approximately as:

$$
O(N^2)
$$


# 18. Hybrid CNN–Transformer

A CNN can first learn local textures; a transformer can then model global relationships.


In [ ]:
class HybridCNNTransformer(nn.Module):
    def __init__(self,dim=64,heads=4,num_classes=3):
        super().__init__()
        self.cnn=nn.Sequential(
            nn.Conv2d(1,32,3,padding=1,stride=2),nn.ReLU(),
            nn.Conv2d(32,dim,3,padding=1,stride=2),nn.ReLU()
        )
        self.block=TransformerBlock(dim,heads)
        self.head=nn.Linear(dim,num_classes)

    def forward(self,x):
        x=self.cnn(x)
        tokens=x.flatten(2).transpose(1,2)
        tokens=self.block(tokens)
        return self.head(tokens.mean(dim=1))


In [ ]:
hybrid=HybridCNNTransformer()
print(hybrid(torch.randn(4,1,64,64)).shape)


# 19. Ultrasound Considerations

Transformers can also learn shortcuts:

- Text overlays
- Borders
- Device markers
- Site-specific rendering

Patient grouping and domain-shift evaluation remain essential.


# 20. Pretraining Domain Gap

ImageNet and ultrasound differ substantially.

Potential alternatives:

- Medical-image pretraining
- Ultrasound self-supervised learning
- Domain-specific foundation models


# 21. Common Mistakes

- Large ViT from scratch on tiny data
- Wrong pretrained normalization
- Wrong input size
- Unvalidated grayscale adaptation
- Treating attention as causal explanation


# 22. Research Comparison

Compare under identical folds/seeds:

1. CNN
2. Pretrained ResNet
3. Pretrained ViT
4. Hybrid CNN–Transformer


# 23. Exercises

1. Compute patch counts.
2. Build patch embedding.
3. Implement self-attention.
4. Build a transformer block.
5. Build a small ViT.
6. Repeat grayscale to RGB.
7. Convert ViT to one channel.
8. Freeze the backbone.
9. Create differential learning rates.
10. Compare CNN vs ViT parameter counts.


# 24. Key Takeaways

ViT flow:

$$
\boxed{
Image\rightarrow Patches\rightarrow Tokens\rightarrow Attention\rightarrow Prediction
}
$$

For small ultrasound datasets, pretrained or hybrid approaches are often more practical than large transformers trained from scratch.


# Next Notebook

# 30 — Medical Image Segmentation with PyTorch and U-Net

In the next notebook, we will study pixel-wise prediction, U-Net, Dice/IoU, segmentation losses, mask augmentation, and ultrasound segmentation evaluation.
